# 🔭 Universe screener — find symbols worth backfilling

A **universe** is just a named list of symbols in `config/universes.yaml`. This
notebook builds one in four steps, all free and keyless:

1. **Fetch** every US-listed stock from the Nasdaq screener API (~1 minute).
2. **Screen** down to liquid common stocks (drop warrants/funds/preferreds, apply
   market-cap and price floors).
3. **Pick** the top names by market cap, capped per sector — so a backtested edge
   is never a hidden bet on one sector.
4. **Persist** — write the universe YAML and register the symbols in the database.

Re-running is always safe: the YAML merges, the database registration UPSERTs.
If the (unofficial) Nasdaq endpoint is ever down, you can simply hand-edit
`config/universes.yaml` instead — it is a plain `name: [symbols]` list.

## 1 · Parameters

In [ ]:
UNIVERSE_NAME = "LiquidLeaders"   # the name strategies will refer to
TOP_N         = 120               # universe size
MIN_CAP       = 1.5e9               # market-cap floor in $ (2e9 = $2B)
MIN_PRICE     = 2.0               # price floor in $ (avoids illiquid penny names)
SECTOR_CAP    = 0.25              # no sector may exceed 25% of the universe

## 2 · Set up

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import universe as uv

## 3 · Fetch all US listings

One request per exchange (NASDAQ / NYSE / AMEX). The result has one row per
symbol with name, last price, market cap, sector and industry.

In [ ]:
listed = uv.fetch_listed_stocks()
print(f"{len(listed):,} US listings fetched; "
      f"sector known for {listed['sector'].notna().mean():.0%} of them")
listed.head(5)

## 4 · Screen down to liquid common stocks

Rules (see `universe.screen_candidates`): plain 1–5 letter symbol, name must not
be a warrant/unit/fund/preferred/SPAC, market cap ≥ the floor, price ≥ the floor.

In [ ]:
screened = uv.screen_candidates(listed, min_market_cap=MIN_CAP, min_price=MIN_PRICE)
print(f"{len(listed):,} listings → {len(screened):,} pass the screen")

import plotly.graph_objects as go
from trading_simple.plots import SERIES
sectors = screened["sector"].fillna("Unknown").value_counts()
fig = go.Figure(go.Bar(x=sectors.index, y=sectors.values, marker={"color": SERIES[0]}))
fig.update_layout(title=f"Screen survivors by sector ({len(screened):,} names)",
                  template="plotly_white", height=350)
fig.show()

## 5 · Pick the universe (sector-capped)

Greedy top-N by market cap, skipping any sector already at
`SECTOR_CAP × TOP_N` names.

In [ ]:
picks = uv.liquid_leaders(screened, n=TOP_N, sector_cap=SECTOR_CAP)
frame = screened[screened["symbol"].isin(picks)]
print(f"{UNIVERSE_NAME}: {len(picks)} names across {frame['sector'].nunique()} sectors")
frame.head(15)[["symbol", "name", "market_cap", "sector"]]

## 6 · (Optional) Rank by trading activity

For names that already have daily prices in the database, average **dollar
volume** (price × volume) is the cheap "how active is it really?" lens — the
backfill notebook uses it to decide which names deserve slow 1-minute history.

In [ ]:
dv = uv.rank_by_dollar_volume(engine, picks)
if len(dv):
    print(f"{len(dv)} of {len(picks)} picks already have daily data")
    display(dv.head(10).style.format("{:,.0f}"))
else:
    print("No daily data yet for these names — run 02_backfill.ipynb first, "
          "then this ranking becomes meaningful.")

## 7 · Persist: write the YAML + register the symbols

In [ ]:
path = uv.write_universe(UNIVERSE_NAME, picks)
n = uv.register_securities(engine, frame)
print(f"wrote {UNIVERSE_NAME} ({len(picks)} symbols) to {path}")
print(f"registered {n} securities (name/sector/exchange) in the shared database")

## 8 · A **point-in-time** universe for intraday strategies

Sections 3–7 build a universe out of *today's* listings. That is fine for a
daily strategy screened afresh each run — and quietly fatal for a backtest,
because `tsim subset` / `rank_by_dollar_volume` rank on `now() - 90 days`.
Pin a universe that way and you have chosen its members **using data from
inside (or after) the window you are about to test on**.

That is not hypothetical. `HighOctaneTop20` was pinned on 2026-07-05 by
trailing-90-day dollar volume — April–July 2026 data — and every 1-minute
strategy backtests 2025-07-01 → 2026-04-30. Equal-weight buy-and-hold of that
basket returned **+316% over the 10-month train window (460%/yr)**: SNDK
+2255%, BE +1123%, neither of which even had pre-window daily history (2025
spinoff and IPO). Since `optimize.OBJECTIVES` is benchmark-relative, every
trial scored `excess_cagr ≈ −4.6` and the whole Pareto front was noise. The
universe, not the strategy, decided the result.

This section builds the honest version, in three rules:

1. **Eligibility is about data, not returns** — keep symbols with at least
   `PIT_MIN_SESSIONS` sessions of the intraday bar inside the backtest window.
2. **Ranking uses only the past** — order by median daily dollar volume over a
   window that *closes before the backtest opens*. Liquidity floors are
   measured there too, so a name that was thin back then cannot sneak in on
   the strength of what it became.
3. **Take the top N.** No sector cap: sector balance is an *outcome* to check
   (see the chart below), and forcing it would re-introduce a choice made with
   hindsight.

One bias this **cannot** remove, and the README must say so: the 1-minute
backfill list itself was assembled in 2026, so pool *membership* is still
survivorship-tinged. Only the ranking within the pool is point-in-time.

In [ ]:
PIT_NAME         = "Morning200"   # the name intraday strategies will refer to
PIT_TOP_N        = 200            # universe size
PIT_BAR          = "1m"           # the intraday bar the universe must be tradable on
PIT_START        = "2025-07-01"   # first bar of the BACKTEST window
PIT_END          = "2026-04-30"   # train cutoff (the holdout tail lives after this)
PIT_MIN_SESSIONS = 200            # required PIT_BAR sessions inside [PIT_START, PIT_END]

# The ranking window. RANK_END is EXCLUSIVE and must not cross PIT_START — that
# single line is the whole point-in-time guarantee, and the assert below enforces it.
RANK_START       = "2025-01-01"
RANK_END         = PIT_START
RANK_MIN_DAYS    = 100            # daily bars required in the ranking window
PIT_MIN_DOLLAR_VOLUME = 20e6      # $/day floor, measured in the RANKING window
PIT_MIN_PRICE    = 5.0            # $ floor, measured in the RANKING window

In [ ]:
import pandas as pd
from sqlalchemy import text


def pit_universe(*, bar, start, end, min_sessions, rank_start, rank_end, rank_min_days,
                 min_dollar_volume, min_price, top_n):
    """Names tradable on `bar` inside [start, end], ranked by dollar volume BEFORE `start`.

    Plain SQL rather than uv.rank_by_dollar_volume: that helper ranks over `now() - N days`
    and cannot be aimed at a historical window (see the note above). The assert is the whole
    point-in-time guarantee.
    """
    assert rank_end <= start, "the ranking window must close before the backtest opens"
    covered = pd.read_sql(text("""
        SELECT symbol, count(DISTINCT (ts AT TIME ZONE 'UTC')::date) AS sessions
        FROM prices_ohlcv
        WHERE bar = :bar AND rth AND ts >= :start AND ts <= :end
        GROUP BY symbol
    """), engine, params={"bar": bar, "start": start, "end": end}).set_index("symbol")
    eligible = covered.index[covered["sessions"] >= min_sessions].tolist()

    # Median (not mean) dollar volume and price: one halted day or one blow-off session
    # should not decide membership. `close * volume` is what actually changed hands, so
    # the raw close is right here — adj_close would restate history.
    ranked = pd.read_sql(text("""
        SELECT symbol,
               percentile_cont(0.5) WITHIN GROUP (ORDER BY close * volume) AS dollar_volume,
               percentile_cont(0.5) WITHIN GROUP (ORDER BY close)          AS price,
               count(*) AS days
        FROM prices_ohlcv
        WHERE bar = '1d' AND symbol = ANY(:symbols) AND ts >= :start AND ts < :end
        GROUP BY symbol
    """), engine, params={"symbols": eligible, "start": rank_start, "end": rank_end}
    ).set_index("symbol")

    pool = ranked[(ranked["days"] >= rank_min_days)
                  & (ranked["dollar_volume"] >= min_dollar_volume)
                  & (ranked["price"] >= min_price)]
    pit = pool.nlargest(top_n, "dollar_volume")
    print(f"{len(covered):,} symbols have {bar} bars in [{start}, {end}]"
          f"  →  {len(eligible):,} with >= {min_sessions} sessions"
          f"  →  {len(pool):,} clear the ranking-window floors"
          f"  →  keeping the top {len(pit):,}")
    return pit


pit = pit_universe(bar=PIT_BAR, start=PIT_START, end=PIT_END, min_sessions=PIT_MIN_SESSIONS,
                   rank_start=RANK_START, rank_end=RANK_END, rank_min_days=RANK_MIN_DAYS,
                   min_dollar_volume=PIT_MIN_DOLLAR_VOLUME, min_price=PIT_MIN_PRICE,
                   top_n=PIT_TOP_N)
pit.head(10).style.format({"dollar_volume": "{:,.0f}", "price": "{:,.2f}", "days": "{:.0f}"})

In [ ]:
def persist_pit(name, pit):
    """Write the universe YAML and chart its sector mix — an outcome to check, never a cap."""
    path = uv.write_universe(name, list(pit.index))
    sectors = pd.read_sql(
        text("SELECT symbol, sector FROM securities WHERE symbol = ANY(:symbols)"),
        engine, params={"symbols": list(pit.index)}).set_index("symbol")["sector"]
    mix = sectors.reindex(pit.index).fillna("Unknown").value_counts()
    print(f"wrote {name} ({len(pit)} symbols) to {path}")
    print(f"{mix.size} sectors, biggest {mix.iloc[0] / len(pit):.0%} of the universe; "
          f"median ${pit['dollar_volume'].median() / 1e6:,.0f}M a day and "
          f"${pit['price'].median():,.0f} a share in the ranking window")
    fig = go.Figure(go.Bar(x=mix.index, y=mix.values, marker={"color": SERIES[0]}))
    fig.update_layout(title=f"{name} by sector ({len(pit)} names, ranked point-in-time)",
                      template="plotly_white", height=350)
    fig.show()


persist_pit(PIT_NAME, pit)

### 8b · The same construction for a **daily** universe — `Liquid500`

A daily cross-sectional strategy needs breadth more than an intraday scanner does — a
turnover sort or an insider sleeve has nothing to rank on 120 mega-caps — and the database
holds daily bars for ~1,950 names from 2020. The rules are identical; only the bar and the
windows change: eligibility is ≥ `DAILY_MIN_SESSIONS` **daily** sessions inside the train
window, ranking is median dollar volume over the half-year *before* it. Built for
`strategies/turnover_momentum`; point any daily strategy at it with
`data.universe: Liquid500`.

Same caveat, one notch worse: `securities` holds **no delisted names** (every row is
`is_active`), so this pool is today's listings and its 2020–2024 history is survivor-only.
The ranking inside the pool is point-in-time; the pool itself is not.

In [ ]:
DAILY_NAME, DAILY_TOP_N = "Liquid500", 500
DAILY_START, DAILY_END = "2020-07-01", "2024-12-31"    # the backtest's TRAIN window
DAILY_MIN_SESSIONS = 1000                              # of ~1,130 daily sessions inside it
DAILY_RANK_START, DAILY_RANK_END = "2020-01-01", DAILY_START   # closes before the backtest
DAILY_MIN_DOLLAR_VOLUME, DAILY_MIN_PRICE = 50e6, 5.0   # floors, measured in the ranking window

liquid = pit_universe(bar="1d", start=DAILY_START, end=DAILY_END,
                      min_sessions=DAILY_MIN_SESSIONS, rank_start=DAILY_RANK_START,
                      rank_end=DAILY_RANK_END, rank_min_days=RANK_MIN_DAYS,
                      min_dollar_volume=DAILY_MIN_DOLLAR_VOLUME, min_price=DAILY_MIN_PRICE,
                      top_n=DAILY_TOP_N)
persist_pit(DAILY_NAME, liquid)

## Next steps

* **`02_backfill.ipynb`** — load daily (and optionally 1-minute) history for
  this universe.
* Then point any strategy at it: set `data.universe: LiquidLeaders` in
  `strategies/<name>/config.yaml`.